In [2]:
from IPython.core.display import HTML

display(HTML('<h1>Hello, world!</h1>'))
print("Here's a link:")
display(HTML("<a href='http://www.google.com' target='_blank'>www.google.com</a>"))
print("some more printed text ...")
display(HTML('<p>Paragraph text here ...</p>'))

Here's a link:


some more printed text ...


In [6]:
import os
import sys
import torch

module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from onmt import opts, inputters
from onmt.bin import translate
from onmt.inputters.news_dataset import load_dataset_from_jsonl, load_pretrained_tokenizer
from onmt.translate import TranslationBuilder
from onmt.utils.parse import ArgumentParser

def _get_parser():
    parser = ArgumentParser(description='translate.py')

    opts.config_opts(parser)
    opts.translate_opts(parser)
    opts.news_opts(parser)
    return parser


### Load model/translator

In [58]:
config_path = '/zfs1/pbrusilovsky/rum20/sum/OpenNMT-summary/config/train/jsonl_tensor/bart_5th_round/test-bart-cnndm-sum-align1e1-wordnpfrag-nocopy-1gpu.config.yml'

parser = _get_parser()
opt = parser.parse_args(args=['-config', config_path])
setattr(opt, 'tokenizer', 'roberta-base')
setattr(opt, 'special_vocab_path', '/zfs1/pbrusilovsky/rum20/sum/newssum/data/vocab/special_vocab')
setattr(opt, 'cache_dir', '/zfs1/pbrusilovsky/rum20/sum/newssum/tools/pretrain_cache/')
setattr(opt, 'gpu', -1)
setattr(opt, 'beam_size', 1)
setattr(opt, 'batch_size', 1)

tokenizer = load_pretrained_tokenizer(opt.tokenizer, cache_dir=opt.cache_dir,
                                      special_vocab_path=opt.special_vocab_path)

our_translator = translate.build_translator(opt)
tgt_field = dict(translator.fields)["tgt"].base_field if hasattr(dict(translator.fields)["tgt"], 'base_field') else \
    dict(translator.fields)["tgt"]


Loading pretrained vocabulary, dumped to /zfs1/pbrusilovsky/rum20/sum/newssum/tools/pretrain_cache/
Vocab size=50265, base vocab size=50265
Added 163 special tokens
Vocab size=50428, base vocab size=50265
loading archive file /zfs1/pbrusilovsky/rum20/sum/newssum/tools/torchhub/bart.large/
| dictionary: 50264 types
loading archive file /zfs1/pbrusilovsky/rum20/sum/newssum/tools/torchhub/bart.large/
| dictionary: 50264 types


In [61]:
config_path = '/zfs1/pbrusilovsky/rum20/sum/OpenNMT-summary/config/train/jsonl_tensor/bart_5th_round/test-bart-cnndm-sum-noalign-nocopy-1gpu.config.yml'

parser = _get_parser()
opt = parser.parse_args(args=['-config', config_path])
setattr(opt, 'tokenizer', 'roberta-base')
setattr(opt, 'special_vocab_path', '/zfs1/pbrusilovsky/rum20/sum/newssum/data/vocab/special_vocab')
setattr(opt, 'cache_dir', '/zfs1/pbrusilovsky/rum20/sum/newssum/tools/pretrain_cache/')
setattr(opt, 'gpu', -1)
setattr(opt, 'beam_size', 1)
setattr(opt, 'batch_size', 1)

baseline_translator = translate.build_translator(opt)


loading archive file /zfs1/pbrusilovsky/rum20/sum/newssum/tools/torchhub/bart.large/
| dictionary: 50264 types
loading archive file /zfs1/pbrusilovsky/rum20/sum/newssum/tools/torchhub/bart.large/
| dictionary: 50264 types


### Load dataset

In [10]:
dataset = load_dataset_from_jsonl(translator.fields, [opt.src],
                                  translator.fields['src'].pretrained_tokenizer, translator.model_opt)
xlation_builder = TranslationBuilder(dataset, translator.fields)
data_iter = inputters.OrderedIterator(
    dataset=dataset,
    device= torch.device(opt.gpu) if int(opt.gpu) > -1 else torch.device("cpu"),
    batch_size=opt.batch_size,
    batch_size_fn=None,
    train=False,
    sort=False,
    sort_within_batch=True,
    shuffle=False
)
batches = list(data_iter)


Loading tensorized src and tgt: 100it [00:00, 208.67it/s]
Processing news examples w/ single processing (building dynamic_dict): 100it [00:02, 39.45it/s]
Processing data examples: 100%|██████████| 100/100 [00:00<00:00, 36405.73it/s]
/ihome/hdaqing/rum20/anaconda3/envs/kp/lib/python3.7/site-packages/torchtext/data/field.py:359: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  var = torch.tensor(arr, dtype=self.dtype, device=device)


### Translate a data example

In [78]:
batch = batches[5]


print('Translating our BART')
batch_trans = our_translator._translate_random_sampling(
            batch, dataset.src_vocabs,
            max_length=70, min_length=20,
            return_attention=True)
translations = xlation_builder.from_batch(batch_trans)
our_trans = translations[0]


print('Translating baseline BART')
batch_trans = baseline_translator._translate_random_sampling(
            batch, dataset.src_vocabs,
            max_length=70, min_length=20,
            return_attention=True)
translations = xlation_builder.from_batch(batch_trans)
baseline_trans = translations[0]


# Post processing and display
src_tokens = [tgt_field.pretrained_tokenizer.convert_ids_to_tokens([idx.item()])[0] for idx in batch.src[:, 0, 0]][1: ]
src_tokens = [t for t in src_tokens if not t.startswith('[') and not t.endswith(']')]
tgt_tokens = [tgt_field.pretrained_tokenizer.convert_ids_to_tokens([idx.item()])[0] for idx in batch.tgt[:, 0, 0]][1: -1]
tgt_tokens = [t for t in tgt_tokens if not t.startswith('[') and not t.endswith(']')]
our_pred_tokens = our_trans.pred_sents[0]
our_pred_tokens = [t for t in our_pred_tokens if not t.startswith('[') and not t.endswith(']')]
our_head_attns = our_trans.attns[0].detach().cpu() if trans.attns[0].is_cuda else our_trans.attns[0].detach()
our_head_attns = our_head_attns.permute(1, 0, 2) # [tgt_len, head_num, src_len] -> [head_num, tgt_len, src_len]

baseline_pred_tokens = baseline_trans.pred_sents[0]
baseline_pred_tokens = [t for t in baseline_pred_tokens if not t.startswith('[') and not t.endswith(']')]
baseline_head_attns = baseline_trans.attns[0].detach().cpu() if trans.attns[0].is_cuda else baseline_trans.attns[0].detach()
baseline_head_attns = baseline_head_attns.permute(1, 0, 2) # [tgt_len, head_num, src_len] -> [head_num, tgt_len, src_len]

src_text =  tokenizer.convert_tokens_to_string(src_tokens)
tgt_text =  tokenizer.convert_tokens_to_string(tgt_tokens)
our_pred_text = tokenizer.convert_tokens_to_string(our_pred_tokens)
baseline_pred_text = tokenizer.convert_tokens_to_string(baseline_pred_tokens)

print('SRC:\n  %s\n' % src_text)
print('TGT:\n  %s\n' % tgt_text)
print('OUR_PRED:\n %s\n' % our_pred_text)
print('BASELINE_PRED:\n %s\n' % baseline_pred_text)

# output = trans.log(sent_number=0)
# print(output)



Translating our BART
Memray: random_sampling for BART decoder only works for batch=1, because there are some issues about the input shape change (decoder_input, active_seq etc.) after some beams end


Error when converting generated codes to string.
Traceback (most recent call last):
  File "/zfs1/pbrusilovsky/rum20/sum/OpenNMT-summary/onmt/translate/translation.py", line 140, in from_batch
    for n in range(self.n_best)]
  File "/zfs1/pbrusilovsky/rum20/sum/OpenNMT-summary/onmt/translate/translation.py", line 140, in <listcomp>
    for n in range(self.n_best)]
IndexError: list index out of range
ERROR batch idx=5


[]
[]
tensor([[    0, 50269]])
tensor([[0]])
Translating baseline BART
Memray: random_sampling for BART decoder only works for batch=1, because there are some issues about the input shape change (decoder_input, active_seq etc.) after some beams end


Error when converting generated codes to string.
Traceback (most recent call last):
  File "/zfs1/pbrusilovsky/rum20/sum/OpenNMT-summary/onmt/translate/translation.py", line 140, in from_batch
    for n in range(self.n_best)]
  File "/zfs1/pbrusilovsky/rum20/sum/OpenNMT-summary/onmt/translate/translation.py", line 140, in <listcomp>
    for n in range(self.n_best)]
IndexError: list index out of range
ERROR batch idx=5


[]
[]
tensor([[    0, 50269]])
tensor([[0]])


IndexError: list index out of range

In [70]:
# take each head, take the mean of all tgt step, and normalize it
src_token_attns = our_head_attns.sum(axis=1) # [head_num, src_len]
for head_id in range(all_head_attns.shape[0]):
    src_token_attns[head_id] = src_token_attns[head_id] / src_token_attns[head_id].max()
    
src_token_attns = src_token_attns.permute(1, 0) # [src_len, head_num]
# print(src_token_attn)
# src_token_attn = src_token_attn.numpy().tolist()
# print(src_token_attn)

display(HTML('<h4>PRED</h4>'))
display(HTML('<p>%s</p>' % (pred_text)))

src_tokens = [tgt_field.pretrained_tokenizer.convert_ids_to_tokens([idx.item()])[0] for idx in batch.src[:, 0, 0]]
src_tokens_mask = [not t.startswith('[') and not t.endswith(']') and not t.startswith('<') for t in src_tokens]
print(len(src_tokens))

html_str = '<div>'
for t, t_attn, t_mask in zip(src_tokens, src_token_attns, src_tokens_mask):
    if t_mask:
        html_str += '<span style="background-color:rgba(255, 0, 0, %f)"><span style="background-color:rgba(0, 255, 0, %f)"><span style="background-color:rgba(0, 0, 255, %f)">%s</span></span></span>' \
        % (0, 0, t_attn[2], t.replace('Ġ', ' '))
        #         % (t_attn[0], t_attn[1], t_attn[2], t.replace('Ġ', ' '))

html_str += '</div>'

display(HTML(html_str))


730


In [72]:
# take each head, take the mean of all tgt step, and normalize it
src_token_attns = baseline_head_attns.sum(axis=1) # [head_num, src_len]
for head_id in range(all_head_attns.shape[0]):
    src_token_attns[head_id] = src_token_attns[head_id] / src_token_attns[head_id].max()
    
src_token_attns = src_token_attns.permute(1, 0) # [src_len, head_num]
# print(src_token_attn)
# src_token_attn = src_token_attn.numpy().tolist()
# print(src_token_attn)

display(HTML('<h4>PRED</h4>'))
display(HTML('<p>%s</p>' % (pred_text)))

src_tokens = [tgt_field.pretrained_tokenizer.convert_ids_to_tokens([idx.item()])[0] for idx in batch.src[:, 0, 0]]
src_tokens_mask = [not t.startswith('[') and not t.endswith(']') and not t.startswith('<') for t in src_tokens]
print(len(src_tokens))

html_str = '<div>'
for t, t_attn, t_mask in zip(src_tokens, src_token_attns, src_tokens_mask):
    if t_mask:
        html_str += '<span style="background-color:rgba(255, 0, 0, %f)"><span style="background-color:rgba(0, 255, 0, %f)"><span style="background-color:rgba(0, 0, 255, %f)">%s</span></span></span>' \
        % (t_attn[0], t_attn[1], t_attn[2], t.replace('Ġ', ' '))
        #         % (0, 0, t_attn[2], t.replace('Ġ', ' '))
                

html_str += '</div>'

display(HTML(html_str))


730
